In [1]:
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent.parent
sys.path.insert(0, str(BASE))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from src.data import Data
import src.metrics as metrics

d:\venv\time-series-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [19]:
## Get the train data.
# Check if it already exists
if not os.path.exists(r"../../../data/single-daywise-sales-list-train.pkl"):
    print("Premade training data not found, cooking it...")
    data = Data()

    prod_id_list = data.get_prod_id_list()
    NUM_DATA = 100
    
    d_train = pd.DataFrame({"Id": prod_id_list.iloc[0:NUM_DATA],
                        "Day_Wise_Sales": [np.array(data.get_train_data(prod, choice1='train', choice2='single_prod_day_wise'))[0] for prod in tqdm(prod_id_list.iloc[0:NUM_DATA])]                   
                        })
    # Since loading data is slow, store it in pkl
    print("Saving training data for quicker future use")
    d_train.to_pickle("../../../data/single-daywise-sales-list-train.pkl")
    print("Training Data saved successfully")
else:
    print("Premade training data found!, loading it...")
    d_train = pd.read_pickle("../../../data/single-daywise-sales-list-train.pkl")
    print("Training Data loaded successfully")


## Similarly get the val data.
if not os.path.exists(r"../../../data/single-daywise-sales-list-val.pkl"):
    print("Premade validation data not found, cooking it...")
    data = Data()

    prod_id_list = data.get_prod_id_list()
    NUM_DATA = 100
    
    d_val = pd.DataFrame({"Id": prod_id_list.iloc[0:NUM_DATA],
                        "Day_Wise_Sales": [np.array(data.get_train_data(prod, choice1='validation', choice2='single_prod_day_wise'))[0] for prod in tqdm(prod_id_list.iloc[0:NUM_DATA])]                   
                        })
    # Since loading data is slow, store it in pkl
    print("Saving validation data for quicker future use")
    d_val.to_pickle("../../../data/single-daywise-sales-list-val.pkl")
    print("Validation Data saved successfully")
else:
    print("Premade validation data found!, loading it...")
    d_val = pd.read_pickle("../../../data/single-daywise-sales-list-val.pkl")
    print("Validation Data loaded successfully")


Premade training data found!, loading it...
Training Data loaded successfully
Premade validation data found!, loading it...
Validation Data loaded successfully


In [21]:
d_train = d_train[d_train['Day_Wise_Sales'].apply(len) > 0]

In [23]:
d_train['Min'] = d_train['Day_Wise_Sales'].apply(np.min)
d_train['Max'] = d_train['Day_Wise_Sales'].apply(np.max)
d_train['Normalized_Day_Wise_Sales'] = (d_train['Day_Wise_Sales'] - d_train['Min'])/ (d_train['Max']  - d_train['Min'])

d_val['Min'] = d_val['Day_Wise_Sales'].apply(np.min)
d_val['Max'] = d_val['Day_Wise_Sales'].apply(np.max)
d_val['Normalized_Day_Wise_Sales'] = (d_val['Day_Wise_Sales'] - d_val['Min'])/ (d_val['Max']  - d_val['Min'])

In [40]:
ids = []
sales = []
time_idx = []

NUM_DATA = 100
data = Data()

for id, norm_sales in tqdm(d_train[: NUM_DATA][['Id', 'Normalized_Day_Wise_Sales']].itertuples(index=False)):

    # Find the week when this products sale begun
    broken_id = id.split('_')
    item_id = broken_id[0] + '_' + broken_id[1] + '_' + broken_id[2]
    store_id = broken_id[3] + '_' + broken_id[4]

    product_wm_yr_wk = data.sell_prices.loc[(data.sell_prices['store_id'] == store_id) & (data.sell_prices['item_id'] == item_id)]['wm_yr_wk'].iloc[0]

    # Find the day from which it's sale would have begun [Assumption: Sale started the very first day of the week it has a sale_price]
    start_day = data.calendar.index[data.calendar['wm_yr_wk'] == product_wm_yr_wk][0] # The day number = the row number

    # y = range(start_day, start_day + len(norm_sales))
    # x = norm_sales
    n = len(norm_sales)
    ids.extend([id] * n)
    sales.extend(norm_sales)
    time_idx.extend(list(range(start_day, start_day + n)))

    pass

train_df = pd.DataFrame({'Id': ids,
                         'Time': time_idx,
                         'value': sales})

Data found in 'data' folder
Loading data...
Data loaded succesfully!


100it [00:14,  7.12it/s]


In [41]:
ids = []
sales = []
time_idx = []

NUM_DATA = 100

for id, norm_sales in tqdm(d_val[: NUM_DATA][['Id', 'Normalized_Day_Wise_Sales']].itertuples(index=False)):

    # Find the week when this products sale begun
    broken_id = id.split('_')
    item_id = broken_id[0] + '_' + broken_id[1] + '_' + broken_id[2]
    store_id = broken_id[3] + '_' + broken_id[4]

    product_wm_yr_wk = data.sell_prices.loc[(data.sell_prices['store_id'] == store_id) & (data.sell_prices['item_id'] == item_id)]['wm_yr_wk'].iloc[0]

    # Find the day from which it's sale would have begun [Assumption: Sale started the very first day of the week it has a sale_price]
    start_day = data.calendar.index[data.calendar['wm_yr_wk'] == product_wm_yr_wk][0] # The day number = the row number

    # y = range(start_day, start_day + len(norm_sales))
    # x = norm_sales
    n = len(norm_sales)
    ids.extend([id] * n)
    sales.extend(norm_sales)
    time_idx.extend(list(range(start_day, start_day + n)))

    pass

val_df = pd.DataFrame({'Id': ids,
                         'Time': time_idx,
                         'value': sales})

Data found in 'data' folder
Loading data...
Data loaded succesfully!


100it [00:10,  9.79it/s]


In [46]:
from pytorch_forecasting import TimeSeriesDataSet, NBeats

WINDOW_SIZE = 32
HORIZON = 28
BATCH_SIZE = 64

training_dataset = TimeSeriesDataSet(train_df, time_idx='Time', target='value', group_ids=['Id'],
                             max_encoder_length=WINDOW_SIZE, max_prediction_length=HORIZON, time_varying_unknown_reals=["value"])

validation_dataset = TimeSeriesDataSet(val_df, time_idx='Time', target='value', group_ids=['Id'],
                             max_encoder_length=WINDOW_SIZE, max_prediction_length=HORIZON, time_varying_unknown_reals=["value"])

train_dataloader = training_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE)

In [49]:
BACKCAST_LOSS_RATIO = 0.1
LR = 1e-3
model = NBeats.from_dataset(training_dataset, learning_rate= LR, backcast_loss_ratio=BACKCAST_LOSS_RATIO)

d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [53]:
from lightning.pytorch import Trainer

EPOCHS = 10

trainer = Trainer(max_epochs=EPOCHS, accelerator=device.type)

trainer.fit(model, train_dataloader, val_dataloader)

GPU available: False, used: False


TPU available: False, using: 0 TPU cores
d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registr

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00, 13.43it/s]

d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\venv\time-series-venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 9: 100%|██████████| 2236/2236 [33:35<00:00,  1.11it/s, v_num=0, train_loss_step=0.745, val_loss=0.712, train_loss_epoch=0.713]  

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 2236/2236 [33:35<00:00,  1.11it/s, v_num=0, train_loss_step=0.745, val_loss=0.712, train_loss_epoch=0.713]


In [ ]:
results = model.predict(val_dataloader, return_y=True)
predictions = results.output.numpy()
actuals = results.y[0].numpy()

In [75]:
metric_names = [
    'MAE',
    'RMSE',
    'WAPE',
    'WRMSE',
    'mean_error'
    ]
scores = {m : [] for m in metric_names}

for met in metric_names:
        scores[met].append(getattr(metrics, met)(predictions, actuals))

    
results_df = pd.DataFrame({
    "Metric": metric_names,
    "Average Score": [
        np.mean(scores[met])
        for met in metric_names
    ]
})

print(results_df.to_string(index=False))

    Metric  Average Score
       MAE       0.083607
      RMSE       0.154322
      WAPE       2.601786
     WRMSE     320.933663
mean_error       0.059037
